In [17]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [18]:
df = pd.read_csv("data/german_credit.csv")

df["target"] = df["default"].map({
    1: 0,
    2: 1
})

df.head()

,checking_balance,months_loan_duration,credit_history,purpose,amount,savings_balance,employment_length,installment_rate,personal_status,other_debtors,...,age,installment_plan,housing,existing_credits,job,dependents,telephone,foreign_worker,default,target
0,< 0 DM,6,critical,radio/tv,1169,unknown,> 7 yrs,4,single male,none,...,67,none,own,2,skilled employee,1,yes,yes,1,0
1,1 - 200 DM,48,repaid,radio/tv,5951,< 100 DM,1 - 4 yrs,2,female,none,...,22,none,own,1,skilled employee,1,none,yes,2,1
2,unknown,12,critical,education,2096,< 100 DM,4 - 7 yrs,2,single male,none,...,49,none,own,1,unskilled resident,2,none,yes,1,0
3,< 0 DM,42,repaid,furniture,7882,< 100 DM,4 - 7 yrs,2,single male,guarantor,...,45,none,for free,1,skilled employee,2,none,yes,1,0
4,< 0 DM,24,delayed,car (new),4870,< 100 DM,1 - 4 yrs,3,single male,none,...,53,none,for free,2,skilled employee,2,none,yes,2,1


In [19]:
X = df.drop(
    columns=["default", "target"]
)

y = df["target"]

X.shape, y.shape

((1000, 20), (1000,))

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape

((800, 20), (200, 20))

In [21]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numeric:")
print(numeric_features)

print("\nCategorical:")
print(categorical_features)

Numeric:
['months_loan_duration', 'amount', 'installment_rate', 'residence_history', 'age', 'existing_credits', 'dependents']

Categorical:
['checking_balance', 'credit_history', 'purpose', 'savings_balance', 'employment_length', 'personal_status', 'other_debtors', 'property', 'installment_plan', 'housing', 'job', 'telephone', 'foreign_worker']


In [22]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [23]:
model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000
        )
    )
])

In [24]:
model.fit(
    X_train,
    y_train
)

d:\Anaconda\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
d:\Anaconda\Lib\site-packages\sklearn\compose\_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, us

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['checking_balance',
                                                   'credit_history', 'purpose',
                                                   'savings_balance',
                                                   'employment_length',
                                                   'personal_status',
                                                   'other_debtors', 'property',
                                                   'installment_plan',
                                                   'housing', 'job',
                                                   'telephone',
                                                   'foreign_worker'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

In [25]:
y_pred = model.predict(X_test)

In [26]:
print(
    "Accuracy:",
    accuracy_score(y_test, y_pred)
)

print(
    "Precision:",
    precision_score(y_test, y_pred)
)

print(
    "Recall:",
    recall_score(y_test, y_pred)
)

print(
    "F1:",
    f1_score(y_test, y_pred)
)

Accuracy: 0.78
Precision: 0.6538461538461539
Recall: 0.5666666666666667
F1: 0.6071428571428571


In [27]:
cm = confusion_matrix(
    y_test,
    y_pred
)

print(cm)

[[122  18]
 [ 26  34]]


In [28]:
y_prob = model.predict_proba(X_test)[:, 1]

y_prob[:10]

array([0.23717163, 0.098476  , 0.60035722, 0.54256191, 0.15269745,
       0.1145529 , 0.22133536, 0.14098661, 0.50847836, 0.15194011])

In [29]:
y_pred_40 = (y_prob >= 0.40).astype(int)

In [30]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("Accuracy:", accuracy_score(y_test, y_pred_40))
print("Precision:", precision_score(y_test, y_pred_40))
print("Recall:", recall_score(y_test, y_pred_40))
print("F1:", f1_score(y_test, y_pred_40))

Accuracy: 0.785
Precision: 0.6268656716417911
Recall: 0.7
F1: 0.6614173228346457


In [31]:
print(
    confusion_matrix(
        y_test,
        y_pred_40
    )
)

[[115  25]
 [ 18  42]]


## Threshold Optimization

The default threshold of 0.5 resulted in a recall score of 56.7%.

By lowering the threshold to 0.4, recall improved to 70.0% while maintaining similar overall accuracy.

For credit risk assessment, this trade-off is beneficial because detecting risky applicants is more important than maximizing overall accuracy.

In [1]:
import sklearn
print(sklearn.__version__)

1.6.1
